In [1]:
RANDOM_STATE = 42
OUT_DIR = "runs"
RUN_NAME = "blte"

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from scipy.stats import randint, uniform
import matplotlib.pyplot as plt
import seaborn as sns


# ==========================
# 0. Cấu hình chung
# ==========================

DATA_PATH = r"D:\elliptic\blte\Labeled-Transactions-based-Dataset-of-Ethereum-Network-master\FinalDataset.xlsx"
# (hoặc .xlsx nếu bạn dùng bản excel -> dùng read_excel)

In [3]:
# ==========================
# 1. Load final_dataset
# ==========================
df = pd.read_excel(DATA_PATH)

print("Số dòng ban đầu:", len(df))
print("Các cột:", df.columns.tolist())

Số dòng ban đầu: 71250
Các cột: ['hash', 'nonce', 'transaction_index', 'from_address', 'to_address', 'value', 'gas', 'gas_price', 'input', 'receipt_cumulative_gas_used', 'receipt_gas_used', 'block_timestamp', 'block_number', 'block_hash', 'from_scam', 'to_scam', 'from_category', 'to_category']


In [4]:
df['rcpt_cum_gas_used'] = df['receipt_cumulative_gas_used']
df = df.drop(columns='receipt_cumulative_gas_used')

In [5]:
# ==========================
# 2. Tạo nhãn transaction-level
# ==========================
# Điền thiếu cho from_scam/to_scam (nếu có NaN)
for col in ["from_scam", "to_scam"]:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# 1 = abnormal nếu from hoặc to là scam
df["label"] = (
    (df.get("from_scam", 0) == 1) |
    (df.get("to_scam", 0) == 1)
).astype(int)

LABEL_COL = "label"

print("Phân bố nhãn (0=normal, 1=abnormal):")
print(df[LABEL_COL].value_counts())

y = df[LABEL_COL].values

Phân bố nhãn (0=normal, 1=abnormal):
label
0    57000
1    14250
Name: count, dtype: int64


In [6]:
# ==========================
# 3. Chọn feature (tránh rò rỉ label)
# ==========================

# (A) Lấy cột thời gian tách riêng (để split), KHÔNG đưa vào feature
ts_col = None
for c in ["block_timestamp", "block_number"]:
    if c in df.columns:
        ts_col = c
        break
if ts_col is None:
    raise ValueError("Không tìm thấy cột thời gian (block_timestamp hoặc block_number).")

ts_raw = df[ts_col].copy()

# Chuẩn hoá ts -> unix seconds (nếu là datetime string như ảnh của bạn)
# Nếu là số (block_number) thì giữ nguyên numeric
if ts_col == "block_timestamp":
    ts_dt = pd.to_datetime(ts_raw, errors="coerce")
    # nếu có NaT, ta ffill/bfill để không vỡ split
    ts_dt = ts_dt.fillna(method="ffill").fillna(method="bfill")
    ts_num = (ts_dt.view("int64") // 10**9).astype("int64")
else:
    ts_num = pd.to_numeric(ts_raw, errors="coerce")
    ts_num = ts_num.fillna(method="ffill").fillna(method="bfill").astype("int64")


# (B) Các cột KHÔNG dùng làm feature:
drop_id_cols = [
    # 'block_number',
    'block_timestamp',   # vẫn drop khỏi feature (đúng ý bạn)
    'txId',

    "hash",
    'transaction_index',
    "from_address",
    "to_address",
    "block_hash",
    "input"
]

drop_label_cols = [
    "from_scam", "to_scam",
    "from_category", "to_category",
    LABEL_COL
]

cols_to_drop = [c for c in drop_id_cols + drop_label_cols if c in df.columns]

feature_candidates = [c for c in df.columns if c not in cols_to_drop]
X_df = df[feature_candidates].copy()

# Giữ lại cột numeric để dùng cho ML
non_numeric = X_df.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print("Bỏ cột không phải số:", non_numeric)
    X_df = X_df.drop(columns=non_numeric)

feature_cols = X_df.columns.tolist()
print("Feature dùng để train:", feature_cols)

X = X_df.values

# Giữ txId để align với preds từ GNN
if "txId" in df.columns:
    txid = df.loc[X_df.index, "txId"].astype(int).values
else:
    txid = np.arange(len(X_df), dtype=int)
print("txid shape:", txid.shape)

# Quan trọng: ts_num phải align đúng với X (cùng index X_df)
ts_num = ts_num.loc[X_df.index].to_numpy()


Feature dùng để train: ['nonce', 'value', 'gas', 'gas_price', 'receipt_gas_used', 'block_number', 'rcpt_cum_gas_used']
txid shape: (71250,)


C:\Users\Admin\AppData\Local\Temp\ipykernel_32872\18801085.py:21: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  ts_dt = ts_dt.fillna(method="ffill").fillna(method="bfill")
C:\Users\Admin\AppData\Local\Temp\ipykernel_32872\18801085.py:22: FutureWarning: Series.view is deprecated and will be removed in a future version. Use ``astype`` as an alternative to change the dtype.
  ts_num = (ts_dt.view("int64") // 10**9).astype("int64")


In [7]:
# ==========================
# 3b. Kiểm tra tương quan feature với label
# ==========================

# Ở đây ta dùng chính X_df (sau khi đã drop ID/label, xử lý timestamp, bỏ non-numeric)
# và cột nhãn LABEL_COL trong df.

# Gộp X_df (feature) với cột label thành một DataFrame chung
corr_df = pd.concat(
    [
        X_df.reset_index(drop=True),
        df[LABEL_COL].reset_index(drop=True)
    ],
    axis=1
)

# Tính ma trận tương quan Pearson
corr_matrix = corr_df.corr()

# Lấy vector tương quan của từng feature với label
corr_with_label = corr_matrix[LABEL_COL].drop(labels=[LABEL_COL])  # bỏ chính label

# Giá trị tuyệt đối để xem mức độ mạnh/yếu
corr_with_label_abs = corr_with_label.abs().sort_values(ascending=False)

print("\nTop 30 feature có |corr| lớn nhất với label:")
print(corr_with_label_abs.head(30))

# Nếu muốn xem cả dấu và |corr| dưới dạng bảng:
corr_table = pd.DataFrame({
    "corr": corr_with_label,
    "abs_corr": corr_with_label_abs
}).sort_values("abs_corr", ascending=False)

corr_table.head(30)



Top 30 feature có |corr| lớn nhất với label:
block_number         0.442214
nonce                0.120640
rcpt_cum_gas_used    0.110460
receipt_gas_used     0.105110
gas_price            0.048455
gas                  0.022670
value                0.015716
Name: label, dtype: float64


,corr,abs_corr
block_number,0.442214,0.442214
nonce,-0.120640,0.120640
rcpt_cum_gas_used,0.110460,0.110460
receipt_gas_used,0.105110,0.105110
gas_price,-0.048455,0.048455
gas,0.022670,0.022670
value,-0.015716,0.015716


In [8]:
# ==========================
# 4. Chia train / val / test THEO THỜI GIAN (70/15/15) - KHÔNG XÁO TRỘN
# ==========================
# Ý tưởng giống Elliptic (Tree): chia theo "time bucket"
# tất cả transaction thuộc cùng 1 time sẽ nằm trong cùng 1 split.

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-9

# Ở cell 3 bạn đã tạo:
#   ts_col  : tên cột thời gian (block_timestamp hoặc block_number)
#   ts_num  : numpy array unix seconds hoặc block_number, đã align với X/y/txid
# Nên ở đây KHÔNG cần đọc df/parse lại nữa.

unique_ts = np.sort(np.unique(ts_num))
n_ts = len(unique_ts)

print(f"ts_col = {ts_col} | số time buckets = {n_ts}")
print("time đầu/cuối:", unique_ts[:5], "...", unique_ts[-5:])

# Cắt theo tỉ lệ time buckets (không phải theo số dòng)
train_cut = int(np.floor(TRAIN_RATIO * n_ts))
val_cut   = int(np.floor((TRAIN_RATIO + VAL_RATIO) * n_ts))

# Đảm bảo mỗi split có ít nhất 1 time bucket
train_cut = max(train_cut, 1)
val_cut   = max(val_cut, train_cut + 1)
val_cut   = min(val_cut, n_ts - 1)

train_ts = unique_ts[:train_cut]
val_ts   = unique_ts[train_cut:val_cut]
test_ts  = unique_ts[val_cut:]

train_mask = np.isin(ts_num, train_ts)
val_mask   = np.isin(ts_num, val_ts)
test_mask  = np.isin(ts_num, test_ts)

# Sanity check: không overlap & cover hết
assert not np.any(train_mask & val_mask)
assert not np.any(train_mask & test_mask)
assert not np.any(val_mask & test_mask)
assert np.all(train_mask | val_mask | test_mask)

X_train, y_train, txid_train = X[train_mask], y[train_mask], txid[train_mask]
X_val,   y_val,   txid_val   = X[val_mask],   y[val_mask],   txid[val_mask]
X_test,  y_test,  txid_test  = X[test_mask],  y[test_mask],  txid[test_mask]

def show_stats(name, yy):
    counts = np.bincount(yy)
    n0 = counts[0] if len(counts) > 0 else 0
    n1 = counts[1] if len(counts) > 1 else 0
    ratio = n1 / (n0 + n1) if (n0 + n1) > 0 else 0
    print(f"{name:5s}: 0 = {n0:6d}, 1 = {n1:6d}, scam_ratio = {ratio:.6f}")

print("\n=== Phân bố nhãn sau khi chia (theo time) ===")
show_stats("ALL",   y)
show_stats("Train", y_train)
show_stats("Val",   y_val)
show_stats("Test",  y_test)

print("\nKích thước:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val  :", X_val.shape,   "y_val  :", y_val.shape)
print("X_test :", X_test.shape,  "y_test :", y_test.shape)


ts_col = block_timestamp | số time buckets = 28683
time đầu/cuối: [1508131613 1508131729 1508131759 1508131783 1508131817] ... [1570405665 1570406197 1570406223 1570406280 1570406377]

=== Phân bố nhãn sau khi chia (theo time) ===
ALL  : 0 =  57000, 1 =  14250, scam_ratio = 0.200000
Train: 0 =  42538, 1 =   3021, scam_ratio = 0.066310
Val  : 0 =   7200, 1 =    699, scam_ratio = 0.088492
Test : 0 =   7262, 1 =  10530, scam_ratio = 0.591839

Kích thước:
X_train: (45559, 7) y_train: (45559,)
X_val  : (7899, 7) y_val  : (7899,)
X_test : (17792, 7) y_test : (17792,)


In [9]:
print("Feature dùng để train:", feature_cols)
print("Các cột nghi ngờ (scam/category/label):")
print([c for c in feature_cols
       if any(k in c.lower() for k in ["scam", "category", "label"])])

Feature dùng để train: ['nonce', 'value', 'gas', 'gas_price', 'receipt_gas_used', 'block_number', 'rcpt_cum_gas_used']
Các cột nghi ngờ (scam/category/label):
[]


In [10]:
# ==========================
# 5. Chuẩn hóa feature
# ==========================
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

In [11]:
# =========================================
# 5b. Load GNN/SAGE TEST predictions để ensemble chung
#     (tree chỉ cần file .npz nên không cần import torch/pyg ở đây)
# =========================================
import os
import numpy as np

def load_saved_proba(save_dirs, filename, txid_test, fill=0.5):
    """
    Load npz {txid, proba, (optional) threshold} và align theo txid_test.
    Trả về: (proba_aligned or None, threshold, used_file)
    """
    for d in save_dirs:
        f = os.path.join(d, filename)
        if os.path.exists(f):
            npz = np.load(f, allow_pickle=True)
            txid = npz["txid"].astype(int)
            proba = npz["proba"].astype(float)
            thr = float(npz["threshold"]) if "threshold" in npz.files else 0.5

            mp = {int(t): float(p) for t, p in zip(txid.tolist(), proba.tolist())}
            aligned = np.array([mp.get(int(t), np.nan) for t in txid_test], dtype=float)

            missing = int(np.isnan(aligned).sum())
            if missing > 0:
                print(f"[WARN] Missing {missing}/{len(aligned)} txId in {f}. Fill={fill}")
                aligned = np.nan_to_num(aligned, nan=fill)

            return aligned, thr, f

    return None, 0.5, None


# ----- GCN/GNN preds (từ txs-gcn) -----
GNN_SAVE_CANDIDATES = [
    "gnn_saved",
    r"D:\elliptic\blte\blte\gnn_saved",
]
gnn_proba_test, gnn_threshold, gnn_file = load_saved_proba(
    GNN_SAVE_CANDIDATES, "gnn_test_preds.npz", txid_test, fill=0.5
)
if gnn_proba_test is not None:
    print("[OK] Loaded GNN preds:", gnn_proba_test.shape, "| thr:", gnn_threshold, "| file:", gnn_file)
else:
    print("[INFO] Không tìm thấy GNN preds trong:", GNN_SAVE_CANDIDATES)


# ----- GraphSAGE preds (từ txs_sage) -----
SAGE_SAVE_CANDIDATES = [
    "sage_saved",
    r"D:\elliptic\blte\blte\sage_saved",
]
sage_proba_test, sage_threshold, sage_file = load_saved_proba(
    SAGE_SAVE_CANDIDATES, "gnn_test_preds.npz", txid_test, fill=0.5
)
if sage_proba_test is not None:
    print("[OK] Loaded SAGE preds:", sage_proba_test.shape, "| thr:", sage_threshold, "| file:", sage_file)
else:
    print("[INFO] Không tìm thấy SAGE preds trong:", SAGE_SAVE_CANDIDATES)


[OK] Loaded GNN preds: (17792,) | thr: 0.5 | file: gnn_saved\gnn_test_preds.npz
[OK] Loaded SAGE preds: (17792,) | thr: 0.5 | file: sage_saved\gnn_test_preds.npz


In [12]:
print("Train size:", X_train_scaled.shape, "Val size:", X_val_scaled.shape, "Test size:", X_test_scaled.shape)

# sanity check: không lẫn y
print("Số nhãn train:", np.bincount(y_train))
print("Số nhãn val  :", np.bincount(y_val))
print("Số nhãn test :", np.bincount(y_test))


Train size: (45559, 7) Val size: (7899, 7) Test size: (17792, 7)
Số nhãn train: [42538  3021]
Số nhãn val  : [7200  699]
Số nhãn test : [ 7262 10530]


In [13]:
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.base import clone
from scipy.stats import randint, uniform

def sample_param(dist, rng):
    """Lấy 1 giá trị từ distribution hoặc list."""
    if hasattr(dist, "rvs"):
        return dist.rvs(random_state=rng)
    # nếu là list/tuple
    dist = list(dist)
    return dist[rng.randint(0, len(dist))]

def random_search_single_model(name, base_estimator, param_dist,
                               X_train, y_train, X_val, y_val,
                               n_iter=30):
    print(f"\n===== Random search cho {name} (không k-fold, dùng VAL) =====")
    rng = np.random.RandomState(RANDOM_STATE)
    best_f1 = -1.0
    best_params = None

    for i in range(n_iter):
        # sample 1 bộ siêu tham số
        params = {k: sample_param(v, rng) for k, v in param_dist.items()}

        model = clone(base_estimator)
        model.set_params(**params)
        model.fit(X_train, y_train)

        y_val_pred = model.predict(X_val)
        f1 = f1_score(y_val, y_val_pred)

        print(f"Iter {i+1:02d}/{n_iter}: F1(val) = {f1:.6f}, params = {params}")

        if f1 > best_f1:
            best_f1 = f1
            best_params = params

    print(f"\n>>> {name} – best F1(val) = {best_f1:.6f}")
    print("Best params:", best_params)

    # Train lại trên TRAIN+VAL với best_params trước khi test
    X_train_full = np.vstack([X_train, X_val])
    y_train_full = np.concatenate([y_train, y_val])

    best_model = clone(base_estimator)
    best_model.set_params(**best_params)
    best_model.fit(X_train_full, y_train_full)

    return best_model


In [14]:
# Tính scale_pos_weight nếu muốn dùng cho XGB/LGBM
n_pos = np.sum(y_train == 1)
n_neg = np.sum(y_train == 0)
scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1.0
print("scale_pos_weight (train):", scale_pos_weight)

scale_pos_weight (train): 14.080767957629924


7. AutoML cho RF / XGB / LGBM

In [15]:

# 1) RandomForest
rf_base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)
rf_param_dist = {
    "n_estimators": randint(200, 600),
    "max_depth": randint(3, 30),
    "min_samples_split": randint(2, 50),
    "min_samples_leaf": randint(1, 20),
    "max_features": ["sqrt", "log2", None],
    "class_weight": [None, "balanced"],
    # thêm criterion vào hyperparameter search
    "criterion": ["gini", "entropy"],  # nếu sklearn mới có thể thêm "log_loss"
}


best_rf = random_search_single_model(
    "RandomForest",
    rf_base,
    rf_param_dist,
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    n_iter=30
)


===== Random search cho RandomForest (không k-fold, dùng VAL) =====
Iter 01/30: F1(val) = 0.764940, params = {'n_estimators': 302, 'max_depth': 22, 'min_samples_split': 30, 'min_samples_leaf': 15, 'max_features': None, 'class_weight': 'balanced', 'criterion': 'gini'}
Iter 02/30: F1(val) = 0.758733, params = {'n_estimators': 220, 'max_depth': 9, 'min_samples_split': 20, 'min_samples_leaf': 11, 'max_features': None, 'class_weight': 'balanced', 'criterion': 'gini'}
Iter 03/30: F1(val) = 0.646660, params = {'n_estimators': 299, 'max_depth': 10, 'min_samples_split': 25, 'min_samples_leaf': 3, 'max_features': 'log2', 'class_weight': None, 'criterion': 'entropy'}
Iter 04/30: F1(val) = 0.709793, params = {'n_estimators': 543, 'max_depth': 14, 'min_samples_split': 31, 'min_samples_leaf': 6, 'max_features': 'log2', 'class_weight': 'balanced', 'criterion': 'entropy'}
Iter 05/30: F1(val) = 0.539185, params = {'n_estimators': 476, 'max_depth': 3, 'min_samples_split': 13, 'min_samples_leaf': 12, 'm

In [16]:
# 2) XGBoost
xgb_base = XGBClassifier(
    random_state=RANDOM_STATE,
    tree_method="hist",      # hoặc "gpu_hist" nếu bạn chạy GPU
    eval_metric="logloss",
    use_label_encoder=False
)
xgb_param_dist = {
    "n_estimators": randint(300, 800),
    "max_depth": randint(3, 10),
    "learning_rate": uniform(0.01, 0.2),
    "subsample": uniform(0.6, 0.4),         # [0.6, 1.0]
    "colsample_bytree": uniform(0.6, 0.4),  # [0.6, 1.0]
    "min_child_weight": randint(1, 10),
    "gamma": uniform(0, 5),
    "reg_lambda": uniform(0, 5),
    "objective": ["binary:logistic"],       # thường giữ nguyên cái này
    "eval_metric": ["logloss", "aucpr"],    # cho AutoML thử 2 metric
}

best_xgb = random_search_single_model(
    "XGBoost",
    xgb_base,
    xgb_param_dist,
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    n_iter=30
)


===== Random search cho XGBoost (không k-fold, dùng VAL) =====


d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [03:40:20] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Iter 01/30: F1(val) = 0.685446, params = {'n_estimators': 402, 'max_depth': 6, 'learning_rate': np.float64(0.20014286128198325), 'subsample': np.float64(0.892797576724562), 'colsample_bytree': np.float64(0.8394633936788146), 'min_child_weight': 7, 'gamma': np.float64(2.229163764267956), 'reg_lambda': np.float64(0.4998745790900144), 'objective': 'binary:logistic', 'eval_metric': 'logloss'}


d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [03:40:20] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Iter 02/30: F1(val) = 0.657061, params = {'n_estimators': 387, 'max_depth': 7, 'learning_rate': np.float64(0.13022300234864176), 'subsample': np.float64(0.8832290311184181), 'colsample_bytree': np.float64(0.608233797718321), 'min_child_weight': 2, 'gamma': np.float64(3.609993861334124), 'reg_lambda': np.float64(4.692763545078751), 'objective': 'binary:logistic', 'eval_metric': 'aucpr'}


...

d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [03:40:33] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Iter 29/30: F1(val) = 0.681132, params = {'n_estimators': 772, 'max_depth': 5, 'learning_rate': np.float64(0.2047327673510635), 'subsample': np.float64(0.7135683898949863), 'colsample_bytree': np.float64(0.7221455441377573), 'min_child_weight': 2, 'gamma': np.float64(3.459475988463466), 'reg_lambda': np.float64(3.259806297513003), 'objective': 'binary:logistic', 'eval_metric': 'logloss'}


d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [03:40:33] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Iter 30/30: F1(val) = 0.630137, params = {'n_estimators': 783, 'max_depth': 3, 'learning_rate': np.float64(0.013615072723104174), 'subsample': np.float64(0.7975574860733738), 'colsample_bytree': np.float64(0.6715290836885315), 'min_child_weight': 8, 'gamma': np.float64(3.2481644952360735), 'reg_lambda': np.float64(4.24611705247089), 'objective': 'binary:logistic', 'eval_metric': 'logloss'}

>>> XGBoost – best F1(val) = 0.732496
Best params: {'n_estimators': 716, 'max_depth': 9, 'learning_rate': np.float64(0.19734599774734693), 'subsample': np.float64(0.6550083776583973), 'colsample_bytree': np.float64(0.7364265404201034), 'min_child_weight': 9, 'gamma': np.float64(1.0453581036885684), 'reg_lambda': np.float64(2.707239869137829), 'objective': 'binary:logistic', 'eval_metric': 'aucpr'}


d:\elliptic\venv1\Lib\site-packages\xgboost\core.py:158: UserWarning: [03:40:34] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [17]:
# 3) LightGBM
lgbm_base = LGBMClassifier(
    random_state=RANDOM_STATE,
    objective="binary",
    n_jobs=-1
)
lgbm_param_dist = {
    "n_estimators": randint(300, 800),
    "num_leaves": randint(15, 255),
    "max_depth": randint(-1, 12),            # -1 = no limit
    "learning_rate": uniform(0.01, 0.2),
    "subsample": uniform(0.6, 0.4),          # bagging_fraction
    "colsample_bytree": uniform(0.6, 0.4),   # feature_fraction
    "min_child_samples": randint(10, 100),
    "reg_lambda": uniform(0, 5),
    # --------- "criterion" kiểu LightGBM: objective + metric ----------
    "objective": ["binary"],                      # có thể thêm "xentropy"
    "metric": ["binary_logloss", "auc", "aucpr"]  # AutoML tự chọn
}

best_lgbm = random_search_single_model(
    "LightGBM",
    lgbm_base,
    lgbm_param_dist,
    X_train_scaled, y_train,
    X_val_scaled,   y_val,
    n_iter=30
)


===== Random search cho LightGBM (không k-fold, dùng VAL) =====
[LightGBM] [Info] Number of positive: 3021, number of negative: 42538
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009404 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1719
[LightGBM] [Info] Number of data points in the train set: 45559, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.066310 -> initscore=-2.644810
[LightGBM] [Info] Start training from score -2.644810
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Iter 03/30: F1(val) = 0.733154, params = {'n_estimators': 685, 'num_leaves': 206, 'max_depth': 10, 'learning_rate': np.float64(0.046680901970686764), 'subsample': np.float64(0.7216968971838151), 'colsample_bytree': np.float64(0.8099025726528951), 'min_child_samples': 98, 'reg_lambda': np.float64(1.4561457009902097), 'objective': 'binary', 'metric': 'aucpr'}
[LightGBM] [Info] Number of positive: 3021, number of negative: 42538
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000244 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1719
[LightGBM] [Info] Number of data points in the train set: 45559, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.066310 -> initscore=-2.644810
[LightGBM] [Info] Start training from score -2.644810
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Iter 29/30: F1(val) = 0.718552, params = {'n_estimators': 703, 'num_leaves': 166, 'max_depth': 4, 'learning_rate': np.float64(0.1659751091715248), 'subsample': np.float64(0.8568126584617151), 'colsample_bytree': np.float64(0.6336559859980195), 'min_child_samples': 81, 'reg_lambda': np.float64(3.926703255569718), 'objective': 'binary', 'metric': 'auc'}
[LightGBM] [Info] Number of positive: 3021, number of negative: 42538
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000250 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1719
[LightGBM] [Info] Number of data points in the train set: 45559, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.066310 -> initscore=-2.644810
[LightGBM] [Info] Start training from score -2.644810
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [War

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [18]:
# =========================================================
# 7. ENSEMBLE (Voting) từ 3 model: 4 tổ hợp × (hard + soft)
#    - Không cần refit lại model (dùng best_rf/best_xgb/best_lgbm đã fit)
# =========================================================
from itertools import combinations
from sklearn.metrics import roc_auc_score

base_models = {
    "RF": best_rf,
    "XGB": best_xgb,
    "LGBM": best_lgbm
}



# Nếu có GNN preds (loaded ở cell 5b) thì thêm vào ensemble như 1 base-model nữa
class StaticProbaModel:
    """Model giả lập sklearn cho ensemble (có predict_proba + predict)."""
    def __init__(self, proba_pos, threshold=0.5):
        self.proba_pos = np.asarray(proba_pos, dtype=float).ravel()
        self.threshold = float(threshold)

    def predict_proba(self, X):
        n = len(X)
        if n != len(self.proba_pos):
            raise ValueError(f"Length mismatch: X has {n} rows but proba_pos has {len(self.proba_pos)}")
        p = self.proba_pos
        return np.vstack([1 - p, p]).T

    def predict(self, X):
        # dùng threshold nội bộ (mặc định 0.5) để phục vụ hard-voting
        n = len(X)
        if n != len(self.proba_pos):
            raise ValueError(f"Length mismatch: X has {n} rows but proba_pos has {len(self.proba_pos)}")
        return (self.proba_pos >= self.threshold).astype(int)


if gnn_proba_test is not None:
    base_models["GNN"] = StaticProbaModel(gnn_proba_test)
    print("[OK] Added GNN into base_models for ensemble.")
if sage_proba_test is not None:
    base_models["SAGE"] = StaticProbaModel(sage_proba_test)
    print("[OK] Added SAGE into base_models for ensemble.")


def _proba_pos(model, X):
    proba = model.predict_proba(X)
    if proba.ndim == 2 and proba.shape[1] >= 2:
        return proba[:, 1]
    return proba.ravel()

def ensemble_predict(models_dict, X, voting="soft", threshold=0.5):
    models = list(models_dict.values())

    probas = np.vstack([_proba_pos(m, X) for m in models])  # [n_models, n_samples]
    proba_mean = probas.mean(axis=0)

    if voting == "soft":
        y_pred = (proba_mean >= threshold).astype(int)
        return y_pred, proba_mean

    if voting == "hard":
        preds = np.vstack([m.predict(X) for m in models]).astype(int)
        votes = preds.sum(axis=0)
        half = len(models) / 2

        y_pred = (votes > half).astype(int)

        # tie (chỉ xảy ra khi 2 model): dùng mean proba để phá hoà
        tie_mask = (votes == half)
        if np.any(tie_mask):
            y_pred[tie_mask] = (proba_mean[tie_mask] >= threshold).astype(int)

        return y_pred, proba_mean

    raise ValueError("voting phải là 'hard' hoặc 'soft'")

def eval_binary(y_true, y_pred, y_score=None):
    out = {}
    out["accuracy"] = accuracy_score(y_true, y_pred)
    out["f1_binary"] = f1_score(y_true, y_pred)
    out["f1_micro"]  = f1_score(y_true, y_pred, average="micro")
    out["f1_macro"]  = f1_score(y_true, y_pred, average="macro")
    if y_score is not None:
        try:
            out["roc_auc"] = roc_auc_score(y_true, y_score)
        except Exception:
            out["roc_auc"] = np.nan
    return out

results = []

# 4 tổ hợp: C(3,2)=3 + C(3,3)=1
from itertools import combinations

model_order = [k for k in ["RF", "XGB", "LGBM", "GNN","SAGE"] if k in base_models]

combos = []
for r in range(1, len(model_order) + 1):  # <-- chỉ lấy từ 2 đến N
    combos.extend(list(combinations(model_order, r)))

print("Total combos =", len(combos))  # 4 model => C(4,2)+C(4,3)+C(4,4)=6+4+1=11

for combo in combos:
    combo_models = {k: base_models[k] for k in combo}
    combo_name = "+".join(combo)

    for voting in ["hard", "soft"]:
        y_pred, y_score = ensemble_predict(combo_models, X_test_scaled, voting=voting, threshold=0.5)
        m = eval_binary(y_test, y_pred, y_score)
        
        results.append({"ensemble": combo_name, "voting": voting, **m})

        print(f"\n===== Ensemble [{combo_name}] | {voting.upper()} vote =====")
        print(classification_report(y_test, y_pred, digits=6))
        print(f"Accuracy: {m['accuracy']:.6f}")
        print(f"F1 (binary, pos_label=1): {m['f1_binary']:.6f}")
        print(f"F1 micro: {m['f1_micro']:.6f}")
        print(f"F1 macro: {m['f1_macro']:.6f}")
        ra = m.get("roc_auc", np.nan)
        print(f"ROC-AUC: {ra:.6f}" if np.isfinite(ra) else "ROC-AUC: nan")
        print("Confusion matrix:")
        print(confusion_matrix(y_test, y_pred))

results_df = pd.DataFrame(results).sort_values(
    by=["f1_binary", "roc_auc", "accuracy"],
    ascending=False
)
display(results_df)


[OK] Added GNN into base_models for ensemble.
[OK] Added SAGE into base_models for ensemble.
Total combos = 31

===== Ensemble [RF] | HARD vote =====
              precision    recall  f1-score   support

           0   0.468460  0.973561  0.632549      7262
           1   0.928889  0.238177  0.379138     10530

    accuracy                       0.538332     17792
   macro avg   0.698675  0.605869  0.505844     17792
weighted avg   0.740960  0.538332  0.482571     17792

Accuracy: 0.538332
F1 (binary, pos_label=1): 0.379138
F1 micro: 0.538332
F1 macro: 0.505844
ROC-AUC: 0.809253
Confusion matrix:
[[7070  192]
 [8022 2508]]

===== Ensemble [RF] | SOFT vote =====
              precision    recall  f1-score   support

           0   0.468460  0.973561  0.632549      7262
           1   0.928889  0.238177  0.379138     10530

    accuracy                       0.538332     17792
   macro avg   0.698675  0.605869  0.505844     17792
weighted avg   0.740960  0.538332  0.482571     17792

Ac

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



===== Ensemble [LGBM] | SOFT vote =====
              precision    recall  f1-score   support

           0   0.682990  0.977554  0.804146      7262
           1   0.977967  0.687085  0.807117     10530

    accuracy                       0.805643     17792
   macro avg   0.830479  0.832319  0.805632     17792
weighted avg   0.857569  0.805643  0.805905     17792

Accuracy: 0.805643
F1 (binary, pos_label=1): 0.807117
F1 micro: 0.805643
F1 macro: 0.805632
ROC-AUC: 0.941384
Confusion matrix:
[[7099  163]
 [3295 7235]]

===== Ensemble [GNN] | HARD vote =====
              precision    recall  f1-score   support

           0   0.508631  0.912972  0.653299      7262
           1   0.867143  0.391738  0.539674     10530

    accuracy                       0.604485     17792
   macro avg   0.687887  0.652355  0.596486     17792
weighted avg   0.720812  0.604485  0.586051     17792

Accuracy: 0.604485
F1 (binary, pos_label=1): 0.539674
F1 micro: 0.604485
F1 macro: 0.596486
ROC-AUC: 0.740998


d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



===== Ensemble [RF+LGBM] | HARD vote =====
              precision    recall  f1-score   support

           0   0.617614  0.983063  0.758621      7262
           1   0.980266  0.580247  0.728986     10530

    accuracy                       0.744661     17792
   macro avg   0.798940  0.781655  0.743804     17792
weighted avg   0.832246  0.744661  0.741082     17792

Accuracy: 0.744661
F1 (binary, pos_label=1): 0.728986
F1 micro: 0.744661
F1 macro: 0.743804
ROC-AUC: 0.920758
Confusion matrix:
[[7139  123]
 [4420 6110]]

===== Ensemble [RF+LGBM] | SOFT vote =====
              precision    recall  f1-score   support

           0   0.617614  0.983063  0.758621      7262
           1   0.980266  0.580247  0.728986     10530

    accuracy                       0.744661     17792
   macro avg   0.798940  0.781655  0.743804     17792
weighted avg   0.832246  0.744661  0.741082     17792

Accuracy: 0.744661
F1 (binary, pos_label=1): 0.728986
F1 micro: 0.744661
F1 macro: 0.743804
ROC-AUC: 0.

...


===== Ensemble [XGB+LGBM+GNN+SAGE] | HARD vote =====
              precision    recall  f1-score   support

           0   0.674904  0.989121  0.802346      7262
           1   0.988950  0.671415  0.799819     10530

    accuracy                       0.801090     17792
   macro avg   0.831927  0.830268  0.801082     17792
weighted avg   0.860768  0.801090  0.800850     17792

Accuracy: 0.801090
F1 (binary, pos_label=1): 0.799819
F1 micro: 0.801090
F1 macro: 0.801082
ROC-AUC: 0.975370
Confusion matrix:
[[7183   79]
 [3460 7070]]

===== Ensemble [XGB+LGBM+GNN+SAGE] | SOFT vote =====
              precision    recall  f1-score   support

           0   0.682306  0.989259  0.807599      7262
           1   0.989261  0.682336  0.807621     10530

    accuracy                       0.807610     17792
   macro avg   0.835783  0.835798  0.807610     17792
weighted avg   0.863974  0.807610  0.807612     17792

Accuracy: 0.807610
F1 (binary, pos_label=1): 0.807621
F1 micro: 0.807610
F1 macro: 

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(



===== Ensemble [RF+XGB+LGBM+GNN+SAGE] | HARD vote =====
              precision    recall  f1-score   support

           0   0.572258  0.991325  0.725632      7262
           1   0.987913  0.488984  0.654174     10530

    accuracy                       0.694020     17792
   macro avg   0.780085  0.740154  0.689903     17792
weighted avg   0.818258  0.694020  0.683340     17792

Accuracy: 0.694020
F1 (binary, pos_label=1): 0.654174
F1 micro: 0.694020
F1 macro: 0.689903
ROC-AUC: 0.969786
Confusion matrix:
[[7199   63]
 [5381 5149]]

===== Ensemble [RF+XGB+LGBM+GNN+SAGE] | SOFT vote =====
              precision    recall  f1-score   support

           0   0.638568  0.989948  0.776350      7262
           1   0.988828  0.613580  0.757267     10530

    accuracy                       0.767199     17792
   macro avg   0.813698  0.801764  0.766808     17792
weighted avg   0.845865  0.767199  0.765056     17792

Accuracy: 0.767199
F1 (binary, pos_label=1): 0.757267
F1 micro: 0.767199
F1 m

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


,ensemble,voting,accuracy,f1_binary,f1_micro,f1_macro,roc_auc
18,XGB+LGBM,hard,0.812893,0.814406,0.812893,0.812881,0.939547
19,XGB+LGBM,soft,0.812893,0.814406,0.812893,0.812881,0.939547
43,XGB+LGBM+GNN,soft,0.811769,0.813291,0.811769,0.811757,0.945239
45,XGB+LGBM+SAGE,soft,0.809184,0.809366,0.809184,0.809184,0.975361
59,XGB+LGBM+GNN+SAGE,soft,0.807610,0.807621,0.807610,0.807610,0.975370
...,...,...,...,...,...,...,...
17,RF+SAGE,soft,0.558228,0.411412,0.558228,0.528918,0.943742
14,RF+GNN,hard,0.548393,0.398713,0.548393,0.518559,0.837180
15,RF+GNN,soft,0.548393,0.398713,0.548393,0.518559,0.837180
0,RF,hard,0.538332,0.379138,0.538332,0.505844,0.809253


In [19]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

models = {
    "RandomForest": best_rf,
    "XGBoost": best_xgb,
    "LightGBM": best_lgbm
}

for name, model in models.items():
    print(f"\n===== {name} trên TEST =====")
    y_pred_test = model.predict(X_test_scaled)

    # score liên tục cho ROC-AUC
    y_score = None
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X_test_scaled)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            y_score = proba[:, 1]
        else:
            y_score = proba.ravel()
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test_scaled)

    acc      = accuracy_score(y_test, y_pred_test)
    f1_scam  = f1_score(y_test, y_pred_test)                    # pos_label=1 mặc định
    f1_micro = f1_score(y_test, y_pred_test, average="micro")
    f1_macro = f1_score(y_test, y_pred_test, average="macro")

    if y_score is not None and len(np.unique(y_test)) >= 2:
        auc = roc_auc_score(y_test, y_score)
    else:
        auc = np.nan

    print(f"Accuracy: {acc:.6f}")
    print(f"F1 (scam – binary, pos_label=1): {f1_scam:.6f}")
    print(f"F1 micro: {f1_micro:.6f}")
    print(f"F1 macro: {f1_macro:.6f}")
    print(f"ROC-AUC: {auc:.6f}" if np.isfinite(auc) else "ROC-AUC: nan")

    print("\nclassification_report:")
    print(classification_report(y_test, y_pred_test, digits=6))

    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred_test))



===== RandomForest trên TEST =====
Accuracy: 0.538332
F1 (scam – binary, pos_label=1): 0.379138
F1 micro: 0.538332
F1 macro: 0.505844
ROC-AUC: 0.809253

classification_report:
              precision    recall  f1-score   support

           0   0.468460  0.973561  0.632549      7262
           1   0.928889  0.238177  0.379138     10530

    accuracy                       0.538332     17792
   macro avg   0.698675  0.605869  0.505844     17792
weighted avg   0.740960  0.538332  0.482571     17792

Confusion matrix:
[[7070  192]
 [8022 2508]]

===== XGBoost trên TEST =====
Accuracy: 0.786421
F1 (scam – binary, pos_label=1): 0.782634
F1 micro: 0.786421
F1 macro: 0.786356
ROC-AUC: 0.931207

classification_report:
              precision    recall  f1-score   support

           0   0.659686  0.984715  0.790078      7262
           1   0.984033  0.649668  0.782634     10530

    accuracy                       0.786421     17792
   macro avg   0.821860  0.817191  0.786356     17792
weighte

d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [20]:
# # SHAP cho 3 model: RF, XGB, LGBM (CHỈ SHAP VALUE)
# import shap
# import numpy as np
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt

# shap.initjs()

# models = {
#     "RandomForest": best_rf,
#     "XGBoost": best_xgb,
#     "LightGBM": best_lgbm
# }

# n_background = min(200, X_train_scaled.shape[0])
# n_explain    = min(200, X_test_scaled.shape[0])

# X_background = X_train_scaled[:n_background]
# X_explain    = X_test_scaled[:n_explain]

# mean_abs_dict = {}

# for name, model in models.items():
#     print(f"-- xử lý SHAP cho {name} --")
#     expl = shap.TreeExplainer(
#         model,
#         data=X_background,
#         feature_perturbation="interventional",
#         model_output="probability"
#     )
#     sv = expl.shap_values(X_explain, check_additivity=False)

#     # chuẩn hoá về sv_pos: (n_samples, n_features) cho class=1
#     if isinstance(sv, list):
#         cls_idx = int(np.where(model.classes_ == 1)[0][0]) if hasattr(model, "classes_") else 1
#         sv_pos = np.array(sv[cls_idx])

#     elif isinstance(sv, np.ndarray) and sv.ndim == 3:
#         cls_idx = int(np.where(model.classes_ == 1)[0][0]) if hasattr(model, "classes_") else (sv.shape[2] - 1)
#         sv_pos = np.array(sv[:, :, cls_idx])

#     else:
#         sv_pos = np.array(sv)

#     assert sv_pos.ndim == 2, f"sv_pos không phải 2D cho model {name} (shape={sv_pos.shape})"

#     mean_abs_dict[name] = np.mean(np.abs(sv_pos), axis=0)  # (n_features,)

# # tạo DataFrame (n_features x n_models)
# df_mean = pd.DataFrame(mean_abs_dict, index=feature_cols)

# # sắp xếp features theo tổng mean across models giảm dần
# df_mean["total"] = df_mean.sum(axis=1)
# df_mean = df_mean.sort_values("total", ascending=False).drop(columns="total")

# # ---- VẼ: Heatmap KHÔNG CHUẨN HOÁ + bar ngang ----
# plt.close("all")
# fig = plt.figure(figsize=(8, 6))
# gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.2], wspace=0.4)


In [21]:
# import seaborn as sns
# import matplotlib.pyplot as plt

# # ================== STYLE SPRINGER (LẤY TỪ CODE MẪU) ==================
# plt.rcParams['font.family'] = 'serif'
# plt.rcParams['font.serif'] = ['Times New Roman', 'DejaVu Serif', 'Liberation Serif']
# plt.rcParams['font.size'] = 12
# plt.rcParams['axes.labelsize'] = 14
# plt.rcParams['axes.titlesize'] = 16
# plt.rcParams['xtick.labelsize'] = 11
# plt.rcParams['ytick.labelsize'] = 11
# plt.rcParams['legend.fontsize'] = 11
# plt.rcParams['figure.titlesize'] = 18

# # ================== VẼ CHO RIÊNG BLTE (1 DÒNG, 2 CỘT) ==================
# plt.close('all')

# # 3 dataset dùng (11, 13)  →  1 dataset ~ 13/3 ≈ 4.3
# fig, axes = plt.subplots(
#     1, 2,
#     figsize=(11, 4.3),     # ✨ kích cỡ tương tự mỗi dòng trong code mẫu
#     dpi=600,
#     constrained_layout=True
# )
# ax0, ax1 = axes

# # ---------- (a) Heatmap: Normalized SHAP ----------
# sns.heatmap(
#     df_mean,
#     annot=True,
#     fmt=".2f",
#     cmap="YlOrRd",
#     linewidths=0.5,
#     linecolor='gray',
#     cbar_kws={"label": "Mean", "shrink": 0.8},
#     ax=ax0,
#     annot_kws={"size": 11}
# )

# ax0.set_title("(a) BLTE ", pad=10, fontweight='bold')
# ax0.set_xlabel("")
# ax0.set_ylabel("")
# ax0.set_yticklabels(df_mean.index, rotation=0)
# ax0.set_xticklabels(df_mean.columns, rotation=0)

# # ---------- (b) Grouped bar: Mean |SHAP| ----------
# # Đưa df_mean về dạng long cho seaborn
# df_long = (
#     df_mean
#     .reset_index()
#     .melt(id_vars="index", var_name="Model", value_name="Importance")
#     .rename(columns={"index": "Feature"})
# )

# # Đảm bảo thứ tự feature giống heatmap (từ trên xuống)
# feature_order = df_mean.index.tolist()

# sns.barplot(
#     data=df_long,
#     x="Importance",
#     y="Feature",
#     hue="Model",
#     order=feature_order,
#     palette=["#1f77b4", "#d62728", "#2ca02c"],   # màu giống code mẫu
#     ax=ax1,
#     edgecolor='black',
#     linewidth=0.8
# )

# ax1.set_title("(b) BLTE ", pad=10, fontweight='bold')
# ax1.set_xlabel("Mean")
# ax1.set_ylabel("")
# ax1.grid(axis="x", linestyle='--', alpha=0.5)

# ax1.legend(
#     title="Model",
#     frameon=True,
#     fancybox=False,
#     edgecolor='black'
# )

# fig.savefig("blte.png", dpi=600, bbox_inches="tight")


In [22]:
from pathlib import Path
import pandas as pd
from sklearn.metrics import f1_score

rows = []

# ===== 3 base models =====
base_models = {"RF": best_rf, "XGB": best_xgb, "LGBM": best_lgbm}
for name, model in base_models.items():
    y_pred = model.predict(X_test_scaled)
    rows.append({
        "seed": int(RANDOM_STATE),
        "model": name,
        "f1_macro": float(f1_score(y_test, y_pred, average="macro"))
    })

# ===== 8 ensemble models =====
if "results_df" in globals():
    tmp = results_df.copy()
    tmp["model"] = tmp["ensemble"].astype(str) + "_" + tmp["voting"].astype(str)
    for _, r in tmp.iterrows():
        rows.append({
            "seed": int(RANDOM_STATE),
            "model": str(r["model"]),
            "f1_macro": float(r["f1_macro"])
        })
else:
    print("WARNING: results_df chưa tồn tại -> chỉ lưu macro-F1 cho 3 base models.")

# ===== LƯU CSV (append) =====
out_csv = Path(r"D:\elliptic\blte\blte\runs\macro_f1_all_models_by_seed.csv")
out_csv.parent.mkdir(parents=True, exist_ok=True)

df_out = pd.DataFrame(rows)

write_header = (not out_csv.exists()) or out_csv.stat().st_size == 0
df_out.to_csv(out_csv, mode="a", header=write_header, index=False)

print("Saved:", out_csv, "| rows:", len(df_out))


Saved: D:\elliptic\blte\blte\runs\macro_f1_all_models_by_seed.csv | rows: 65


d:\elliptic\venv1\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
# import os

# out_dir = "outputs"
# os.makedirs(out_dir, exist_ok=True)

# csv_path = os.path.join(out_dir, "ensemble_results.csv")
# results_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

# print("Saved:", csv_path)

Saved: outputs\ensemble_results.csv
